In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; 
# To disable autoreload; run %autoreload 0

process the DIM user

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os 
import sys
from utils.transformations import reusable
from databricks.sdk.runtime import *
projetc_pth=os.path.join(os.getcwd(),'..','..')
sys.path.append(projetc_pth)

### Autoloader

In [0]:
#  STEP 1 — Read Stream of all 4 dim and fact
#fact
FactStream = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/FactStream/schema") \
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/spotify_catalog/datalake/rawdata/FactStream")

# Dimensions
DimUser = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimUser/schema") \
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/spotify_catalog/datalake/rawdata/DimUser")

DimTrack = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimTrack/schema") \
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/spotify_catalog/datalake/rawdata/DimTrack")
   
DimDate = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimDate/schema") \
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/spotify_catalog/datalake/rawdata/DimDate")

DimArtist = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimArtist/schema") \
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("/Volumes/spotify_catalog/datalake/rawdata/DimArtist")



In [0]:

# #  STEP 2 — Write stream to datalake_itslef
 
# df_fact.writeStream \
#     .option("checkpointLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/FactStream/deltalake_checkpoints") \
#     .outputMode("append") \
#     .trigger(availableNow=True)\
#     .start("abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/FactStream/data")
#     # .table("spotify_catalog.bronze.DimUser")


### Writing to bronze


In [0]:
from pyspark.sql import functions as F

# ── FactStream ──────────────────────────────────────────────────────────────
FactStream = FactStream \
    .withColumn("input_file_path",  F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

FactStream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/FactStream/checkpoints") \
    .trigger(availableNow=True) \
    .table("spotify_catalog.bronze.FactStream")

# ── DimUser ─────────────────────────────────────────────────────────────────
DimUser = DimUser \
    .withColumn("input_file_path",  F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

DimUser.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimUser/checkpoints") \
    .trigger(availableNow=True) \
    .table("spotify_catalog.bronze.DimUser")

# ── DimArtist ────────────────────────────────────────────────────────────────
DimArtist = DimArtist \
    .withColumn("input_file_path",  F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

DimArtist.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimArtist/checkpoints") \
    .trigger(availableNow=True) \
    .table("spotify_catalog.bronze.DimArtist")

# ── DimTrack ─────────────────────────────────────────────────────────────────
DimTrack = DimTrack \
    .withColumn("input_file_path",  F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

DimTrack.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimTrack/checkpoints") \
    .trigger(availableNow=True) \
    .table("spotify_catalog.bronze.DimTrack")

# ── DimDate ──────────────────────────────────────────────────────────────────
DimDate = DimDate \
    .withColumn("input_file_path",  F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

DimDate.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://deltalake@mkazureproject1.dfs.core.windows.net/bronze/DimDate/checkpoints") \
    .trigger(availableNow=True) \
    .table("spotify_catalog.bronze.DimDate")

### Basic Transformations

In [0]:
# df_user = spark.read.table("spotify_catalog.bronze.DimUser")
# df_user = df_user.drop("_rescued_data")
# df_artist = spark.read.table("spotify_catalog.bronze.DimArtist")
# df_artist = df_artist.drop("_rescued_data")
# df_date=spark.read.table("spotify_catalog.bronze.DimDate")
# df_date=df_date.drop("_rescued_data")
# df_date=spark.read.table("spotify_catalog.bronze.FactStream")
# df_date=df_date.drop("_rescued_data")

In [0]:
# df_track=spark.read.table("spotify_catalog.bronze.DimTrack")
# df_track=df_track.withColumn("durationFlag",when(col("duration_sec")<150,"low")\
#                             .when(col("duration_sec")<=300,"medium")\
#                             .otherwise("high"))

# df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),'-',''))
# df_track=df_track.drop("_rescued_data")